###### H0: conversion rate is the same for ad and PSA groups
###### H1: conversion rates differ
###### Significance level: 0.05, two-sided

In [9]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf

df = pd.read_csv('../data/processed/ab_test_clean.csv')

summary = df.groupby('test_group')['converted'].agg(['count','sum','mean'])
summary.columns = ['users','conversions','conv_rate']
summary

,users,conversions,conv_rate
test_group,,,
ad,564577,14423,0.025547
psa,23524,420,0.017854


In [10]:
conversions = np.array([summary.loc['ad','conversions'], summary.loc['psa','conversions']])
users = np.array([summary.loc['ad','users'], summary.loc['psa','users']])

z_stat, p_value = proportions_ztest(conversions, users)
print('z: ', round(z_stat, 2), '| p-value: ', p_value)

z:  7.37 | p-value:  1.7052807161559727e-13


In [11]:
p_ad = conversions[0] / users[0]
p_psa = conversions[1] / users[1]

abs_lift = p_ad - p_psa
rel_lift = abs_lift / p_psa

print('absolute lift: ', round(abs_lift * 100, 2), 'percentage points')
print('relative lift: ', round(rel_lift * 100, 1), '%')

low, high = confint_proportions_2indep(conversions[0], users[0], conversions[1], users[1], method = 'wald')
print('95% CI for difference: ', round(low * 100, 2), 'to', round(high * 100, 2), 'percentage points')

absolute lift:  0.77 percentage points
relative lift:  43.1 %
95% CI for difference:  0.6 to 0.94 percentage points


In [12]:
table = pd.crosstab(df['test_group'], df['converted'])
chi2, p, dof, expected = chi2_contingency(table, correction=False)
print('chi2: ', round(chi2, 1), '| p-value: ', p)
print('z squared: ', round(z_stat ** 2, 1))

chi2:  54.3 | p-value:  1.7052807161560133e-13
z squared:  54.3


In [13]:
df["converted_int"] = df["converted"].astype(int)
df["is_ad"] = (df["test_group"] == "ad").astype(int)

simple = smf.logit("converted_int ~ is_ad", data=df).fit()
adjusted = smf.logit("converted_int ~ is_ad + C(most_ads_day) + C(most_ads_hour)", data=df).fit()

print("odds ratio, unadjusted:", round(np.exp(simple.params["is_ad"]), 3))
print("odds ratio, adjusted:  ", round(np.exp(adjusted.params["is_ad"]), 3))

Optimization terminated successfully.
         Current function value: 0.117730
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.116987
         Iterations 9
odds ratio, unadjusted: 1.442
odds ratio, adjusted:   1.442


In [14]:
print(np.exp(adjusted.conf_int().loc["is_ad"]))

0    1.307658
1    1.590745
Name: is_ad, dtype: float64
